# Etap II analiza fuzji joint+bone

Trzy analizy robię na gotowych logitach z cache, bez liczenia niczego od nowa. Najpierw sprawdzam, ile z tych +8 punktów procentowych przewagi bone w dokładności ogólnej bierze się z klas interakcji, a ile z klas pojedynczych. Potem robię pełne zestawienie per-class dla wszystkich 51 klas PKU, żeby zobaczyć, które konkretnie klasy bone poprawia, a które psuje. Na koniec testuję ważoną fuzję z parametrem alfa i sprawdzam krzyżowo między protokołami, czy da się znaleźć kompromis lepszy niż czyste joint albo czyste bone.

In [3]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, pickle
import numpy as np

DRIVE_DATA = '/content/drive/MyDrive/PKU_data'
DATASETS = {'xsub': 'PKU_XSub_both.npz', 'xview': 'PKU_XView_both.npz'}
SPLITS = ['xsub', 'xview']
INTERACTION_NTU = [49, 50, 51, 52, 53, 54, 55, 57]

def load_logits(stream, split):
    cache_dir = f'{DRIVE_DATA}/etap2_scores_{stream}/{split}'
    files = [p for p in os.listdir(cache_dir) if p.endswith('score.pkl')]
    assert files, f'Brak score.pkl w {cache_dir}'
    with open(f'{cache_dir}/{files[0]}', 'rb') as f:
        scores = pickle.load(f)
    keys = sorted(scores.keys(), key=lambda x: int(str(x).split('_')[1])
                  if '_' in str(x) else int(x))
    return np.array([scores[k] for k in keys])

LOGITS_J, LOGITS_B, Y_TRUE = {}, {}, {}
for sp in SPLITS:
    npz = f'{DRIVE_DATA}/{DATASETS[sp]}'
    if not os.path.exists(npz):
        npz = f'/content/SkateFormer/data/ntu/{DATASETS[sp]}'
    data = np.load(npz)
    Y_TRUE[sp] = np.argmax(data['y_test'], axis=1)
    LOGITS_J[sp] = load_logits('j', sp)
    LOGITS_B[sp] = load_logits('b', sp)
    print(f'{sp.upper()}: {len(Y_TRUE[sp])} segmentow, '
          f'joint {LOGITS_J[sp].shape}, bone {LOGITS_B[sp].shape}')

Mounted at /content/drive
XSUB: 2696 segmentow, joint (2696, 60), bone (2696, 60)
XVIEW: 7141 segmentow, joint (7141, 60), bone (7141, 60)


Bone poprawia dokładność ogólną o 8 pp, ale psuje interakcje. Ta poprawa musi
więc pochodzić z klas pojedynczych (nie-interakcyjnych).

In [4]:
def top1(logits, y, classes=None, exclude=None):
    yp = np.argmax(logits, axis=1)
    if classes is not None:
        m = np.isin(y, classes)
    elif exclude is not None:
        m = ~np.isin(y, exclude)
    else:
        m = np.ones(len(y), bool)
    if m.sum() == 0: return None, 0
    return (yp[m] == y[m]).mean()*100, int(m.sum())

rows = []
for sp in SPLITS:
    y = Y_TRUE[sp]
    lj, lb = LOGITS_J[sp], LOGITS_B[sp]
    lf = (lj + lb) / 2
    for tag, lg in [('joint', lj), ('bone', lb), ('j+b', lf)]:
        acc_all, n_all = top1(lg, y)
        acc_int, n_int = top1(lg, y, classes=INTERACTION_NTU)
        acc_sng, n_sng = top1(lg, y, exclude=INTERACTION_NTU)
        rows.append({'split': sp, 'stream': tag,
                     'ogolem': round(acc_all,2),
                     'interakcje': round(acc_int,2),
                     'pojedyncze': round(acc_sng,2)})

import pandas as pd
df1 = pd.DataFrame(rows)
print(f'Liczba klas pojedynczych: 51 - 8 = 43')
print(f'Segmenty interakcji: XSub {top1(LOGITS_J["xsub"],Y_TRUE["xsub"],classes=INTERACTION_NTU)[1]}, '
      f'pojedyncze: {top1(LOGITS_J["xsub"],Y_TRUE["xsub"],exclude=INTERACTION_NTU)[1]}')
display(df1)
print('\nJesli bone >> joint na "pojedyncze" a << na "interakcje" -> potwierdzone:')
print('bone to strumien dla akcji pojedynczych, joint dla interakcji.')

Liczba klas pojedynczych: 51 - 8 = 43
Segmenty interakcji: XSub 195, pojedyncze: 2501


,split,stream,ogolem,interakcje,pojedyncze
0,xsub,joint,62.61,84.10,60.94
1,xsub,bone,70.77,76.41,70.33
2,xsub,j+b,70.07,84.10,68.97
3,xview,joint,62.79,89.84,60.44
4,xview,bone,71.32,80.39,70.53
5,xview,j+b,70.52,89.14,68.90



Jesli bone >> joint na "pojedyncze" a << na "interakcje" -> potwierdzone:
bone to strumien dla akcji pojedynczych, joint dla interakcji.


In [5]:
NAMES_INT = {49:'punch/slap',50:'kick',51:'push',52:'pat on back',
             53:'point at',54:'hugging',55:'giving',57:'handshake'}

for sp in SPLITS:
    y = Y_TRUE[sp]
    lj, lb = LOGITS_J[sp], LOGITS_B[sp]
    lf = (lj + lb) / 2
    yj, yb, yf = np.argmax(lj,1), np.argmax(lb,1), np.argmax(lf,1)

    present = sorted(set(y.tolist()))
    per = []
    for c in present:
        cm = (y == c); nc = cm.sum()
        aj = (yj[cm]==c).mean()*100
        ab = (yb[cm]==c).mean()*100
        af = (yf[cm]==c).mean()*100
        per.append({'ntu': c, 'nazwa': NAMES_INT.get(c, f'klasa_{c}'),
                    'n': int(nc), 'joint': round(aj,1), 'bone': round(ab,1),
                    'j+b': round(af,1), 'bone-joint': round(ab-aj,1)})
    dfp = pd.DataFrame(per).sort_values('bone-joint', ascending=False)
    print(f'\n{"="*70}\n{sp.upper()} - per-class (sortowane wg zysku bone nad joint)\n{"="*70}')
    print('KLASY GDZIE BONE NAJBARDZIEJ POMAGA:')
    display(dfp.head(8).reset_index(drop=True))
    print('KLASY GDZIE BONE NAJBARDZIEJ SZKODZI:')
    display(dfp.tail(8).reset_index(drop=True))


XSUB - per-class (sortowane wg zysku bone nad joint)
KLASY GDZIE BONE NAJBARDZIEJ POMAGA:


,ntu,nazwa,n,joint,bone,j+b,bone-joint
0,2,klasa_2,54,31.5,77.8,70.4,46.3
1,10,klasa_10,53,13.2,54.7,35.8,41.5
2,26,klasa_26,54,18.5,59.3,44.4,40.7
3,19,klasa_19,57,59.6,96.5,86.0,36.8
4,20,klasa_20,57,59.6,91.2,87.7,31.6
5,4,klasa_4,54,13.0,44.4,31.5,31.5
6,14,klasa_14,51,74.5,94.1,92.2,19.6
7,36,klasa_36,54,66.7,85.2,77.8,18.5


KLASY GDZIE BONE NAJBARDZIEJ SZKODZI:


,ntu,nazwa,n,joint,bone,j+b,bone-joint
0,21,klasa_21,54,94.4,88.9,96.3,-5.6
1,44,klasa_44,51,64.7,56.9,66.7,-7.8
2,5,klasa_5,63,100.0,92.1,96.8,-7.9
3,49,punch/slap,24,79.2,70.8,79.2,-8.3
4,54,hugging,24,95.8,79.2,91.7,-16.7
5,55,giving,24,95.8,79.2,91.7,-16.7
6,45,klasa_45,54,55.6,31.5,42.6,-24.1
7,53,point at,24,91.7,58.3,87.5,-33.3



XVIEW - per-class (sortowane wg zysku bone nad joint)
KLASY GDZIE BONE NAJBARDZIEJ POMAGA:


,ntu,nazwa,n,joint,bone,j+b,bone-joint
0,20,klasa_20,125,34.4,86.4,76.0,52.0
1,2,klasa_2,143,24.5,74.8,65.0,50.3
2,10,klasa_10,143,15.4,58.0,49.7,42.7
3,19,klasa_19,127,48.8,89.8,81.1,40.9
4,48,klasa_48,141,29.8,68.1,49.6,38.3
5,26,klasa_26,165,23.0,51.5,33.3,28.5
6,0,klasa_0,157,8.3,30.6,16.6,22.3
7,39,klasa_39,143,54.5,73.4,71.3,18.9


KLASY GDZIE BONE NAJBARDZIEJ SZKODZI:


,ntu,nazwa,n,joint,bone,j+b,bone-joint
0,54,hugging,71,97.2,91.5,95.8,-5.6
1,21,klasa_21,140,97.9,92.1,97.9,-5.7
2,5,klasa_5,171,90.1,84.2,87.7,-5.8
3,49,punch/slap,72,91.7,84.7,88.9,-6.9
4,45,klasa_45,129,58.9,41.1,53.5,-17.8
5,11,klasa_11,152,46.7,25.7,38.8,-21.1
6,55,giving,73,93.2,67.1,89.0,-26.0
7,53,point at,72,93.1,55.6,86.1,-37.5


In [6]:
NAMES_PKU = {
    0:'drink water', 1:'eat meal', 2:'brush teeth', 3:'brush hair', 4:'drop',
    5:'pick up', 6:'throw', 7:'sit down', 8:'stand up', 9:'clapping',
    10:'reading', 11:'writing', 12:'tear up paper', 13:'put on jacket',
    14:'take off jacket', 17:'put on glasses', 18:'take off glasses',
    19:'put on hat', 20:'take off hat', 21:'cheer up', 22:'hand waving',
    23:'kicking something', 24:'reach into pocket', 25:'hopping', 26:'jump up',
    27:'phone call', 28:'playing with phone', 29:'typing', 30:'point to something',
    31:'taking selfie', 32:'check time', 33:'rub two hands', 34:'nod head/bow',
    36:'wipe face', 37:'salute', 39:'cross hands', 42:'falling down',
    43:'headache', 44:'stomachache', 45:'back pain', 46:'neck pain', 48:'fan self',
    49:'punch/slap', 50:'kicking person', 51:'pushing person', 52:'pat on back',
    53:'point finger', 54:'hugging', 55:'giving object', 56:'touch pocket',
    57:'shaking hands',
}

def per_class_table(sp):
    y = Y_TRUE[sp]
    lj, lb = LOGITS_J[sp], LOGITS_B[sp]
    lf = (lj + lb) / 2
    yj, yb, yf = np.argmax(lj,1), np.argmax(lb,1), np.argmax(lf,1)
    rows = []
    for c in sorted(set(y.tolist())):
        cm = (y == c); nc = cm.sum()
        if nc == 0: continue
        aj = (yj[cm]==c).mean()*100
        ab = (yb[cm]==c).mean()*100
        af = (yf[cm]==c).mean()*100
        rows.append({'ntu': c, 'akcja': NAMES_PKU.get(c, f'ntu_{c}'),
                     'typ': 'interakcja' if c in INTERACTION_NTU else 'pojedyncza',
                     'n': int(nc), 'joint': round(aj,1), 'bone': round(ab,1),
                     'j+b': round(af,1), 'j-b': round(aj-ab,1)})
    return pd.DataFrame(rows)

import pandas as pd
for sp in SPLITS:
    df = per_class_table(sp)
    print(f'\n{"#"*72}\n#  {sp.upper()} — pelna tabela 51 klas\n{"#"*72}')

    print('\n=== (c) Wszystkie klasy, wg dokladnosci joint ===')
    display(df.sort_values('joint', ascending=False).reset_index(drop=True))

    print('\n=== (a) 10 klas, ktore JOINT rozpoznaje najgorzej ===')
    display(df.sort_values('joint').head(10)[
        ['akcja','typ','n','joint','bone','j+b']].reset_index(drop=True))

    print('\n=== (b) 10 klas, gdzie JOINT najbardziej przewyzsza bone ===')
    top_j = df.sort_values('j-b', ascending=False).head(10)
    display(top_j[['akcja','typ','n','joint','bone','j-b']].reset_index(drop=True))
    n_int = (top_j['typ']=='interakcja').sum()
    print(f'  Z 10 klas gdzie joint dominuje: {n_int} to interakcje '
          f'({n_int/10*100:.0f}%)')

    df.sort_values('joint', ascending=False).to_csv(
        f'{DRIVE_DATA}/etap2_per_class_{sp}.csv', index=False)

print('\nZapisano etap2_per_class_xsub.csv i etap2_per_class_xview.csv')


########################################################################
#  XSUB — pelna tabela 51 klas
########################################################################

=== (c) Wszystkie klasy, wg dokladnosci joint ===


,ntu,akcja,typ,n,joint,bone,j+b,j-b
0,13,put on jacket,pojedyncza,51,100.0,100.0,100.0,0.0
1,8,stand up,pojedyncza,60,100.0,100.0,100.0,0.0
2,5,pick up,pojedyncza,63,100.0,92.1,96.8,7.9
3,42,falling down,pojedyncza,54,100.0,98.1,100.0,1.9
4,6,throw,pojedyncza,54,98.1,98.1,98.1,0.0
5,34,nod head/bow,pojedyncza,51,98.0,98.0,98.0,0.0
6,50,kicking person,interakcja,27,96.3,92.6,92.6,3.7
7,37,salute,pojedyncza,54,96.3,98.1,98.1,-1.9
8,55,giving object,interakcja,24,95.8,79.2,91.7,16.7
9,54,hugging,interakcja,24,95.8,79.2,91.7,16.7



=== (a) 10 klas, ktore JOINT rozpoznaje najgorzej ===


,akcja,typ,n,joint,bone,j+b
0,touch pocket,pojedyncza,143,0.0,1.4,0.0
1,eat meal,pojedyncza,57,1.8,8.8,7.0
2,typing,pojedyncza,54,9.3,13.0,14.8
3,tear up paper,pojedyncza,47,12.8,23.4,17.0
4,drop,pojedyncza,54,13.0,44.4,31.5
5,reading,pojedyncza,53,13.2,54.7,35.8
6,drink water,pojedyncza,60,13.3,23.3,16.7
7,jump up,pojedyncza,54,18.5,59.3,44.4
8,phone call,pojedyncza,54,29.6,35.2,33.3
9,brush teeth,pojedyncza,54,31.5,77.8,70.4



=== (b) 10 klas, gdzie JOINT najbardziej przewyzsza bone ===


...


=== (a) 10 klas, ktore JOINT rozpoznaje najgorzej ===


,akcja,typ,n,joint,bone,j+b
0,touch pocket,pojedyncza,308,0.0,0.3,0.3
1,eat meal,pojedyncza,142,7.0,4.9,4.9
2,drink water,pojedyncza,157,8.3,30.6,16.6
3,drop,pojedyncza,146,15.1,33.6,27.4
4,reading,pojedyncza,143,15.4,58.0,49.7
5,jump up,pojedyncza,165,23.0,51.5,33.3
6,brush teeth,pojedyncza,143,24.5,74.8,65.0
7,headache,pojedyncza,139,26.6,42.4,37.4
8,typing,pojedyncza,139,28.1,35.3,32.4
9,fan self,pojedyncza,141,29.8,68.1,49.6



=== (b) 10 klas, gdzie JOINT najbardziej przewyzsza bone ===


,akcja,typ,n,joint,bone,j-b
0,point finger,interakcja,72,93.1,55.6,37.5
1,giving object,interakcja,73,93.2,67.1,26.0
2,writing,pojedyncza,152,46.7,25.7,21.1
3,back pain,pojedyncza,129,58.9,41.1,17.8
4,punch/slap,interakcja,72,91.7,84.7,6.9
5,pick up,pojedyncza,171,90.1,84.2,5.8
6,cheer up,pojedyncza,140,97.9,92.1,5.7
7,hugging,interakcja,71,97.2,91.5,5.6
8,playing with phone,pojedyncza,169,64.5,59.8,4.7
9,eat meal,pojedyncza,142,7.0,4.9,2.1


  Z 10 klas gdzie joint dominuje: 4 to interakcje (40%)

Zapisano etap2_per_class_xsub.csv i etap2_per_class_xview.csv


In [7]:
for sp in SPLITS:
    y = Y_TRUE[sp]
    lj = LOGITS_J[sp]
    yj = np.argmax(lj, 1)
    mask56 = (y == 56)
    print(f'\n{sp.upper()}: touch pocket (56) — {mask56.sum()} segmentow')
    preds = yj[mask56]
    vals, cnts = np.unique(preds, return_counts=True)
    NAMES = {24:'reach into pocket', 56:'touch pocket'}
    for v, c in sorted(zip(vals, cnts), key=lambda x:-x[1])[:5]:
        print(f'   -> przewiduje klase {v} ({NAMES.get(v,"?")}): {c}x ({c/mask56.sum()*100:.0f}%)')
    print(f'   Etykieta 24 (reach) w tym samym zbiorze: {(y==24).sum()} segmentow')


XSUB: touch pocket (56) — 143 segmentow
   -> przewiduje klase 45 (?): 58x (41%)
   -> przewiduje klase 24 (reach into pocket): 55x (38%)
   -> przewiduje klase 4 (?): 15x (10%)
   -> przewiduje klase 53 (?): 10x (7%)
   -> przewiduje klase 27 (?): 2x (1%)
   Etykieta 24 (reach) w tym samym zbiorze: 147 segmentow

XVIEW: touch pocket (56) — 308 segmentow
   -> przewiduje klase 24 (reach into pocket): 134x (44%)
   -> przewiduje klase 45 (?): 87x (28%)
   -> przewiduje klase 4 (?): 36x (12%)
   -> przewiduje klase 53 (?): 15x (5%)
   -> przewiduje klase 44 (?): 14x (5%)
   Etykieta 24 (reach) w tym samym zbiorze: 306 segmentow


In [8]:
#put/take pocket i reach into pocket to fizycznie ta sama akcja siegania do kieszeni.
for sp in SPLITS:
    y = Y_TRUE[sp].copy()
    y_corr = y.copy()
    y_corr[y_corr == 56] = 24

    lj, lb = LOGITS_J[sp], LOGITS_B[sp]
    lf = (lj + lb) / 2
    def acc_corrected(logits):
        yp = np.argmax(logits, 1).copy()
        yp[yp == 56] = 24
        return (yp == y_corr).mean() * 100

    print(f'\n{sp.upper()} — wplyw korekty touch pocket 56->24:')
    for tag, lg in [('joint', lj), ('bone', lb), ('j+b', lf)]:
        before = (np.argmax(lg,1) == Y_TRUE[sp]).mean()*100
        after = acc_corrected(lg)
        print(f'  {tag:5s}: {before:.2f}% -> {after:.2f}%  (Δ {after-before:+.2f})')


XSUB — wplyw korekty touch pocket 56->24:
  joint: 62.61% -> 64.65%  (Δ +2.04)
  bone : 70.77% -> 75.11%  (Δ +4.34)
  j+b  : 70.07% -> 73.55%  (Δ +3.49)

XVIEW — wplyw korekty touch pocket 56->24:
  joint: 62.79% -> 64.68%  (Δ +1.89)
  bone : 71.32% -> 74.39%  (Δ +3.07)
  j+b  : 70.52% -> 73.23%  (Δ +2.70)


In [9]:
#Nowy Etap II z korekta klas kieszeniowych (56-24)
def eval_corrected(sp):
    y = Y_TRUE[sp].copy()
    y[y == 56] = 24
    lj, lb = LOGITS_J[sp], LOGITS_B[sp]
    lf = (lj + lb) / 2
    out = {}
    for tag, lg in [('joint', lj), ('bone', lb), ('j+b', lf)]:
        yp = np.argmax(lg, 1).copy()
        yp[yp == 56] = 24
        acc_all = (yp == y).mean()*100
        mi = np.isin(y, INTERACTION_NTU)
        acc_int = (yp[mi] == y[mi]).mean()*100
        ms = ~np.isin(y, INTERACTION_NTU)
        acc_sng = (yp[ms] == y[ms]).mean()*100
        out[tag] = (acc_all, acc_int, acc_sng)
    return out

print(f'{"":6}{"OGOLEM":>22}{"INTERAKCJE":>22}{"POJEDYNCZE":>22}')
print(f'{"":6}{"stary":>10}{"nowy":>11}{"stary":>11}{"nowy":>11}{"stary":>11}{"nowy":>11}')
STARE = {
  'xsub': {'joint':(62.61,84.10,60.94),'bone':(70.77,76.41,70.33),'j+b':(70.07,84.10,68.97)},
  'xview':{'joint':(62.79,89.84,60.44),'bone':(71.32,80.39,70.53),'j+b':(70.52,89.14,68.90)},
}
for sp in SPLITS:
    print(f'\n{sp.upper()}')
    corr = eval_corrected(sp)
    for tag in ['joint','bone','j+b']:
        na, ni, ns = corr[tag]
        sa, si, ss = STARE[sp][tag]
        print(f'{tag:6}{sa:>10.2f}{na:>11.2f}{si:>11.2f}{ni:>11.2f}{ss:>11.2f}{ns:>11.2f}')

import pandas as pd
rows=[]
for sp in SPLITS:
    corr=eval_corrected(sp)
    for tag in ['joint','bone','j+b']:
        na,ni,ns=corr[tag]
        rows.append({'split':sp,'stream':tag,'ogolem':round(na,2),
                     'interakcje':round(ni,2),'pojedyncze':round(ns,2)})
pd.DataFrame(rows).to_csv(f'{DRIVE_DATA}/etap2_kanon_skorygowany.csv',index=False)
print('\nZapisano etap2_kanon_skorygowany.csv')

                      OGOLEM            INTERAKCJE            POJEDYNCZE
           stary       nowy      stary       nowy      stary       nowy

XSUB
joint      62.61      64.65      84.10      84.10      60.94      63.13
bone       70.77      75.11      76.41      76.41      70.33      75.01
j+b        70.07      73.55      84.10      84.10      68.97      72.73

XVIEW
joint      62.79      64.68      89.84      89.84      60.44      62.50
bone       71.32      74.39      80.39      80.39      70.53      73.87
j+b        70.52      73.23      89.14      89.14      68.90      71.84

Zapisano etap2_kanon_skorygowany.csv


In [10]:
for sp in SPLITS:
    y = Y_TRUE[sp].copy()
    y[y == 56] = 24
    print(f'\n{"="*56}\n{sp.upper()} — klasa "siegniecie do kieszeni" (24, po scaleniu)\n{"="*56}')
    for tag, lg in [('joint', LOGITS_J[sp]),
                    ('bone', LOGITS_B[sp]),
                    ('j+b', (LOGITS_J[sp]+LOGITS_B[sp])/2)]:
        yp = np.argmax(lg, 1).copy()
        yp[yp == 56] = 24
        cm = (y == 24)
        n = cm.sum()
        recall = (yp[cm] == 24).mean()*100
        pm = (yp == 24)
        precision = (y[pm] == 24).mean()*100 if pm.sum()>0 else 0
        print(f'  {tag:5s}: n={n}  recall={recall:.1f}%  precision={precision:.1f}%')

    y_old = Y_TRUE[sp]
    m56 = (y_old == 56)
    yp_old = np.argmax(LOGITS_J[sp], 1)
    print(f'  [przed korekta] put in pocket (56): n={m56.sum()}, '
          f'recall={(yp_old[m56]==56).mean()*100:.1f}%  <- to bylo ~0%')


XSUB — klasa "siegniecie do kieszeni" (24, po scaleniu)
  joint: n=290  recall=66.2%  precision=66.7%
  bone : n=290  recall=88.6%  precision=79.8%
  j+b  : n=290  recall=80.7%  precision=74.3%
  [przed korekta] put in pocket (56): n=143, recall=0.0%  <- to bylo ~0%

XVIEW — klasa "siegniecie do kieszeni" (24, po scaleniu)
  joint: n=614  recall=59.9%  precision=72.6%
  bone : n=614  recall=80.3%  precision=81.1%
  j+b  : n=614  recall=75.1%  precision=79.3%
  [przed korekta] put in pocket (56): n=308, recall=0.0%  <- to bylo ~0%


# Ważona fuzja α·joint + (1−α)·bone
Sprawdzam ważoną fuzję logitów joint i bone z parametrem alfa. Testuję różne wartości alfa osobno dla dokładności ogólnej i osobno dla klas interakcji, a następnie sprawdzam krzyżowo między protokołami, czy optymalna wartość alfa znaleziona na jednym protokole sprawdza się też na drugim. Celem jest ustalenie, czy istnieje taka wartość alfa, która jednocześnie daje dobrą dokładność ogólną i dobrą dokładność na klasach interakcji.

In [11]:
ALPHAS = np.round(np.arange(0.0, 1.01, 0.1), 2)

curve = {sp: {'all': [], 'int': []} for sp in SPLITS}
for sp in SPLITS:
    y = Y_TRUE[sp]
    for a in ALPHAS:
        lf = a * LOGITS_J[sp] + (1-a) * LOGITS_B[sp]
        curve[sp]['all'].append(top1(lf, y)[0])
        curve[sp]['int'].append(top1(lf, y, classes=INTERACTION_NTU)[0])

tab = pd.DataFrame({'alpha': ALPHAS})
for sp in SPLITS:
    tab[f'{sp}_ogolem'] = [round(x,2) for x in curve[sp]['all']]
    tab[f'{sp}_interakcje'] = [round(x,2) for x in curve[sp]['int']]
print('α=0: czysty bone   α=1: czysty joint')
display(tab)

print('\n── Walidacja krzyzowa (dokladnosc ogolna) ──')
for tr, te in [('xsub','xview'), ('xview','xsub')]:
    bi = int(np.argmax(curve[tr]['all'])); a = ALPHAS[bi]
    m50 = curve[te]['all'][list(ALPHAS).index(0.5)]
    mst = curve[te]['all'][bi]
    print(f'  α*={a} (z {tr.upper()}) -> {te.upper()}: {mst:.2f}% vs 50/50: {m50:.2f}% '
          f'(Δ {mst-m50:+.2f})')

α=0: czysty bone   α=1: czysty joint


,alpha,xsub_ogolem,xsub_interakcje,xview_ogolem,xview_interakcje
0,0.0,70.77,76.41,71.32,80.39
1,0.1,71.03,78.46,71.59,81.96
2,0.2,70.92,78.97,71.78,83.19
3,0.3,70.96,78.46,71.91,84.41
4,0.4,70.81,79.49,71.53,86.34
5,0.5,70.07,84.10,70.52,89.14
6,0.6,68.32,84.62,68.58,90.02
7,0.7,66.58,84.62,66.83,89.67
8,0.8,65.17,84.10,65.44,89.49
9,0.9,63.91,83.59,63.88,89.32



── Walidacja krzyzowa (dokladnosc ogolna) ──
  α*=0.1 (z XSUB) -> XVIEW: 71.59% vs 50/50: 70.52% (Δ +1.06)
  α*=0.3 (z XVIEW) -> XSUB: 70.96% vs 50/50: 70.07% (Δ +0.89)


In [12]:
ALPHAS = np.round(np.arange(0.0, 1.01, 0.1), 2)

def top1_corr(logits, y, classes=None):
    yp = np.argmax(logits, 1).copy(); yp[yp==56] = 24
    m = np.isin(y, classes) if classes is not None else np.ones(len(y), bool)
    return (yp[m] == y[m]).mean()*100 if m.sum() else None

curve = {sp: {'all': [], 'int': [], 'sng': []} for sp in SPLITS}
for sp in SPLITS:
    y = Y_TRUE[sp].copy(); y[y==56] = 24
    for a in ALPHAS:
        lf = a*LOGITS_J[sp] + (1-a)*LOGITS_B[sp]
        curve[sp]['all'].append(top1_corr(lf, y))
        curve[sp]['int'].append(top1_corr(lf, y, INTERACTION_NTU))
        sng = [c for c in set(y.tolist()) if c not in INTERACTION_NTU]
        curve[sp]['sng'].append(top1_corr(lf, y, sng))

import pandas as pd
tab = pd.DataFrame({'alpha': ALPHAS})
for sp in SPLITS:
    tab[f'{sp}_ogolem'] = [round(x,2) for x in curve[sp]['all']]
    tab[f'{sp}_inter'] = [round(x,2) for x in curve[sp]['int']]
    tab[f'{sp}_pojed'] = [round(x,2) for x in curve[sp]['sng']]
print('α=0: czysty bone   α=1: czysty joint  (dane z korekta 56->24)')
display(tab)

for sp in SPLITS:
    a_all = ALPHAS[np.argmax(curve[sp]['all'])]
    a_int = ALPHAS[np.argmax(curve[sp]['int'])]
    a_sng = ALPHAS[np.argmax(curve[sp]['sng'])]
    print(f'{sp.upper()}: optimum α — ogolem={a_all}, interakcje={a_int}, pojedyncze={a_sng}')
print('\nJesli optimum ogolem != optimum interakcje -> brak jednego dobrego α,')
print('fuzja wazona nie rozwiazuje komplementarnosci strumieni.')

α=0: czysty bone   α=1: czysty joint  (dane z korekta 56->24)


,alpha,xsub_ogolem,xsub_inter,xsub_pojed,xview_ogolem,xview_inter,xview_pojed
0,0.0,75.11,76.41,75.01,74.39,80.39,73.87
1,0.1,75.30,78.46,75.05,74.61,81.96,73.97
2,0.2,75.15,78.97,74.85,74.78,83.19,74.05
3,0.3,75.04,78.46,74.77,74.91,84.41,74.08
4,0.4,74.67,79.49,74.29,74.42,86.34,73.38
5,0.5,73.55,84.10,72.73,73.23,89.14,71.84
6,0.6,71.51,84.62,70.49,71.14,90.02,69.50
7,0.7,69.47,84.62,68.29,69.14,89.67,67.35
8,0.8,67.66,84.10,66.37,67.58,89.49,65.68
9,0.9,66.17,83.59,64.81,65.89,89.32,63.85


XSUB: optimum α — ogolem=0.1, interakcje=0.6, pojedyncze=0.1
XVIEW: optimum α — ogolem=0.3, interakcje=0.6, pojedyncze=0.3

Jesli optimum ogolem != optimum interakcje -> brak jednego dobrego α,
fuzja wazona nie rozwiazuje komplementarnosci strumieni.


In [13]:
for train, test in [('xsub','xview'), ('xview','xsub')]:
    a_star = ALPHAS[np.argmax(curve[train]['all'])]
    idx = list(ALPHAS).index(a_star)
    map_star = curve[test]['all'][idx]
    map_bone = curve[test]['all'][0]
    map_5050 = curve[test]['all'][5]
    print(f'α*={a_star} (dobrane na {train.upper()}) -> na {test.upper()}:')
    print(f'   ogolem={map_star:.2f}%  |  czyste bone={map_bone:.2f}%  '
          f'(Δ {map_star-map_bone:+.2f})  |  50/50={map_5050:.2f}% (Δ {map_star-map_5050:+.2f})')

α*=0.1 (dobrane na XSUB) -> na XVIEW:
   ogolem=74.61%  |  czyste bone=74.39%  (Δ +0.22)  |  50/50=73.23% (Δ +1.39)
α*=0.3 (dobrane na XVIEW) -> na XSUB:
   ogolem=75.04%  |  czyste bone=75.11%  (Δ -0.07)  |  50/50=73.55% (Δ +1.48)


In [14]:
df1.to_csv(f'{DRIVE_DATA}/etap2_rozbicie_dokladnosci.csv', index=False)
tab.to_csv(f'{DRIVE_DATA}/etap2_alpha_sweep.csv', index=False)
print('Zapisano:')
print(f'  {DRIVE_DATA}/etap2_rozbicie_dokladnosci.csv')
print(f'  {DRIVE_DATA}/etap2_alpha_sweep.csv')

Zapisano:
  /content/drive/MyDrive/PKU_data/etap2_rozbicie_dokladnosci.csv
  /content/drive/MyDrive/PKU_data/etap2_alpha_sweep.csv


In [16]:
import shutil
import os

os.makedirs('/content/SkateFormer', exist_ok=True)
DRIVE_WEIGHTS = '/content/drive/MyDrive/SkateFormer_weights/ntu60_CSub'
WEIGHTS_LOCAL = {}
for key, fname in [('j', 'SkateFormer_j.pt'), ('b', 'SkateFormer_b.pt')]:
    src = f'{DRIVE_WEIGHTS}/{fname}'
    dst = f'/content/SkateFormer/{fname}'
    if not os.path.exists(dst):
        shutil.copy(src, dst)
    WEIGHTS_LOCAL[key] = dst
    print(f'  {key}: {dst}  ({os.path.getsize(dst)/1024**2:.1f} MB)')

BASELINE_DATASETS = {'xsub': 'PKU_XSub.npz', 'xview': 'PKU_XView.npz'}
os.makedirs('/content/SkateFormer/data/ntu', exist_ok=True)
BASELINE_LOCAL = {}
for split, fname in BASELINE_DATASETS.items():
    src = f'{DRIVE_DATA}/{fname}'
    dst = f'/content/SkateFormer/data/ntu/{fname}'
    if not os.path.exists(dst):
        shutil.copy(src, dst)
    BASELINE_LOCAL[split] = dst
    d = np.load(dst)
    print(f'  {split}: {fname}  test={d["x_test"].shape}')

def build_config(weights_pt, npz_path, data_type, work_dir):
    return {
        'seed': 1, 'num_worker': 4, 'work_dir': work_dir, 'phase': 'test',
        'weights': weights_pt, 'feeder': 'feeders.feeder_ntu.Feeder',
        'train_feeder_args': {
            'data_path': npz_path, 'split': 'train', 'debug': False,
            'window_size': 64, 'p_interval': [0.5, 1], 'aug_method': 'a123489',
            'intra_p': 0.5, 'inter_p': 0.2, 'thres': 64, 'uniform': True,
            'partition': True, 'data_type': data_type
        },
        'test_feeder_args': {
            'data_path': npz_path, 'split': 'test', 'window_size': 64,
            'p_interval': [0.95], 'thres': 64, 'uniform': True,
            'partition': True, 'debug': False, 'data_type': data_type
        },
        'model': 'model.SkateFormer.SkateFormer_',
        'model_args': {
            'num_classes': 60, 'num_people': 2, 'num_points': 24, 'kernel_size': 7,
            'num_heads': 32, 'attn_drop': 0.5, 'head_drop': 0.0, 'rel': True,
            'drop_path': 0.2, 'type_1_size': [8, 8], 'type_2_size': [8, 12],
            'type_3_size': [8, 8], 'type_4_size': [8, 12], 'mlp_ratio': 4.0,
            'index_t': True
        },
        'device': [0], 'batch_size': 128, 'test_batch_size': 128,
        'num_epoch': 1, 'save_score': True
    }

def run_baseline_eval(stream, split, force=False):
    data_type = 'joint' if stream == 'j' else 'b'
    npz_path = BASELINE_LOCAL[split]
    work_dir = f'/content/SkateFormer/work_dir/etap2_baseline_{stream}_{split}'
    cache_dir = f'{DRIVE_DATA}/etap2_baseline_scores_{stream}/{split}'
    os.makedirs(work_dir, exist_ok=True)
    os.makedirs(cache_dir, exist_ok=True)

    cached = [p for p in os.listdir(cache_dir) if p.endswith('score.pkl')]
    if cached and not force:
        with open(f'{cache_dir}/{cached[0]}', 'rb') as f:
            scores = pickle.load(f)
        print(f'  [{stream}/{split}] cache z Drive: pomijam trening')
    else:
        cfg = build_config(WEIGHTS_LOCAL[stream], npz_path, data_type, work_dir)
        cfg_path = f'{work_dir}/config.yaml'
        with open(cfg_path, 'w') as f:
            yaml.dump(cfg, f)
        os.system(f'cd /content/SkateFormer && python main.py --config "{cfg_path}"')
        existing = [p for p in os.listdir(work_dir) if p.endswith('score.pkl')]
        assert existing, f'Brak score.pkl w {work_dir} -- sprawdz log powyzej'
        shutil.copy(f'{work_dir}/{existing[0]}', f'{cache_dir}/{existing[0]}')
        with open(f'{work_dir}/{existing[0]}', 'rb') as f:
            scores = pickle.load(f)

    keys = sorted(scores.keys(), key=lambda x: int(str(x).split('_')[1])
                  if '_' in str(x) else int(x))
    return np.array([scores[k] for k in keys])

def acc_corr(logits, y, classes=None):
    yp = np.argmax(logits, 1).copy(); yp[yp == 56] = 24
    yy = y.copy(); yy[yy == 56] = 24
    m = np.isin(yy, classes) if classes is not None else np.ones(len(yy), bool)
    return (yp[m] == yy[m]).mean() * 100

STARE_BASELINE = {
    'xsub': {'joint': 67.95}, 'xview': {'joint': 67.08},
}

print('\n' + '='*60)
print('BASELINE z korekta 56->24 (uzupelnienie Tabeli 4.12)')
print('='*60)

base_rows = []
for split in SPLITS:
    y_true = np.argmax(np.load(BASELINE_LOCAL[split])['y_test'], axis=1)
    lj = run_baseline_eval('j', split)
    lb = run_baseline_eval('b', split)
    lf = (lj + lb) / 2

    print(f'\n{split.upper()}')
    for tag, lg in [('joint', lj), ('bone', lb), ('j+b', lf)]:
        a_all = acc_corr(lg, y_true)
        a_int = acc_corr(lg, y_true, INTERACTION_NTU)
        sng = [c for c in set(y_true.tolist()) if c not in INTERACTION_NTU]
        a_sng = acc_corr(lg, y_true, sng)
        stary = STARE_BASELINE[split].get(tag)
        stary_str = f'{stary:.2f}%' if stary else '-'
        base_rows.append({'split': split, 'wariant': 'baseline', 'stream': tag,
                           'ogolem_stary': stary, 'ogolem_nowy': round(a_all, 2),
                           'interakcje': round(a_int, 2), 'pojedyncze': round(a_sng, 2)})
        print(f'  {tag:5s}: ogolem {stary_str} -> {a_all:.2f}%  '
              f'interakcje={a_int:.2f}%  pojedyncze={a_sng:.2f}%')

import pandas as pd
df_base = pd.DataFrame(base_rows)
df_base.to_csv(f'{DRIVE_DATA}/etap2_baseline_kanon.csv', index=False)
print('\nZapisano etap2_baseline_kanon.csv')
print('\nJesli ogolem_nowy > ogolem_stary o ok. 2pp (jak przy wariancie both,')
print('62,61% -> 64,65%) -> korekta dziala spojnie niezaleznie od preprocessingu,')
print('Tabela 4.12 wymaga podmiany liczby 67,95%/67,08% na wartosc ogolem_nowy.')

  j: /content/SkateFormer/SkateFormer_j.pt  (13.9 MB)
  b: /content/SkateFormer/SkateFormer_b.pt  (13.9 MB)
  xsub: PKU_XSub.npz  test=(2696, 300, 150)
  xview: PKU_XView.npz  test=(7141, 300, 150)

BASELINE z korekta 56->24 (uzupelnienie Tabeli 4.12)
  [j/xsub] cache z Drive: pomijam trening
  [b/xsub] cache z Drive: pomijam trening

XSUB
  joint: ogolem 67.95% -> 67.95%  interakcje=81.54%  pojedyncze=66.89%
  bone : ogolem - -> 77.67%  interakcje=80.00%  pojedyncze=77.49%
  j+b  : ogolem - -> 75.26%  interakcje=82.56%  pojedyncze=74.69%
  [j/xview] cache z Drive: pomijam trening
  [b/xview] cache z Drive: pomijam trening

XVIEW
  joint: ogolem 67.08% -> 67.08%  interakcje=88.09%  pojedyncze=65.25%
  bone : ogolem - -> 77.30%  interakcje=83.89%  pojedyncze=76.73%
  j+b  : ogolem - -> 75.13%  interakcje=89.14%  pojedyncze=73.91%

Zapisano etap2_baseline_kanon.csv

Jesli ogolem_nowy > ogolem_stary o ok. 2pp (jak przy wariancie both,
62,61% -> 64,65%) -> korekta dziala spojnie niezalezni

In [17]:
for split in SPLITS:
    y_true = np.argmax(np.load(BASELINE_LOCAL[split])['y_test'], axis=1)
    lj = run_baseline_eval('j', split)
    yp_raw = np.argmax(lj, 1)
    print(f'{split}: liczba predykcji==56: {(yp_raw==56).sum()}  '
          f'liczba etykiet==56: {(y_true==56).sum()}  '
          f'liczba etykiet==24: {(y_true==24).sum()}')

  [j/xsub] cache z Drive: pomijam trening
xsub: liczba predykcji==56: 52  liczba etykiet==56: 143  liczba etykiet==24: 147
  [j/xview] cache z Drive: pomijam trening
xview: liczba predykcji==56: 97  liczba etykiet==56: 308  liczba etykiet==24: 306


In [18]:
for split in SPLITS:
    y_true = np.argmax(np.load(BASELINE_LOCAL[split])['y_test'], axis=1)
    lj = run_baseline_eval('j', split)
    yp_raw = np.argmax(lj, 1)

    acc_bez_korekty = (yp_raw == y_true).mean() * 100
    acc_z_korekta = acc_corr(lj, y_true)

    print(f'{split}: bez korekty={acc_bez_korekty:.2f}%  z korekta={acc_z_korekta:.2f}%  '
          f'roznica={acc_z_korekta - acc_bez_korekty:+.2f}pp')

  [j/xsub] cache z Drive: pomijam trening
xsub: bez korekty=65.28%  z korekta=67.95%  roznica=+2.67pp
  [j/xview] cache z Drive: pomijam trening
xview: bez korekty=65.05%  z korekta=67.08%  roznica=+2.03pp


Korekta zmienia wynik o +2,67pp/+2,03pp